# Rust CUDA kernels, second session: the compile cache on Google Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xavierforge/colab-rust/blob/main/examples/05_cuda_oxide_drive_cache.ipynb)

[04_cuda_oxide.ipynb](https://colab.research.google.com/github/xavierforge/colab-rust/blob/main/examples/04_cuda_oxide.ipynb) takes a blank runtime to a running kernel; its slow step is the configuration cell that compiles cuda-rust's crates and everything they depend on, and a fresh runtime starts from nothing. This notebook is the same walk-through with that step cached on your Drive: evcxr's compile cache (`:cache`) stores each dependency's build output under `~/.cache/evcxr`, keyed by the exact rustc command line, and copies it back instead of recompiling. Section 3 restores the cache from Drive, section 6 saves it. Run it once to fill the cache; every later session restores it.

**Needs a GPU runtime** and your consent to mount Drive (section 3 asks). Everything else is as in 04.


## 1. Check the runtime


In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!grep PRETTY_NAME /etc/os-release

## 2. Install

As in 04: `setup-cuda-oxide.sh` runs `setup.sh` first if the session does not have it yet.


In [ ]:
!curl -fsSL -o /tmp/setup-cuda-oxide.sh https://raw.githubusercontent.com/xavierforge/colab-rust/feat/drive-compile-cache/setup-cuda-oxide.sh
!bash /tmp/setup-cuda-oxide.sh
%load_ext colab_rust

## 3. Restore the compiled crates from Google Drive

The configuration cell below compiles cuda-rust's crates and everything they depend on, and a fresh runtime starts from nothing. evcxr has a compile cache (`:cache`) that stores each dependency's build output under `~/.cache/evcxr`, keyed by the exact rustc command line, and copies it back instead of recompiling. This cell mounts your Drive and unpacks the archive that section 6 saved there, so the configuration cell gets cache hits instead of a full build. On the first run there is nothing to restore yet.

The cache travels as one tar archive rather than as a folder on Drive: Drive keeps no executable bit, and a build script copied back from a mounted folder cannot run. The archive is named after the pinned cuda-oxide commit and the PTX target (`cced2a35bcc5-sm_75` on a T4 at the current pin), because the cache key sees neither the target (it travels in the environment) nor the contents of the checkout; a pin bump or another card means one more cold session, and archives of old pins can be deleted by hand. Only dependencies are cached; your own cells compile every time.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib, subprocess, time
oxide = pathlib.Path('/opt/colab-rust/cuda-oxide')
archive = pathlib.Path('/content/drive/MyDrive/colab-rust') / f"evcxr-cache-{oxide.resolve().name[:12]}-{(oxide / 'target').read_text().strip()}.tar"
cache = pathlib.Path.home() / '.cache' / 'evcxr'
cache.mkdir(parents=True, exist_ok=True)
if archive.exists():
    t = time.time()
    subprocess.run(['tar', '-xf', archive, '-C', cache.parent], check=True)
    restored = sum(1 for _ in cache.iterdir())
    print(f"restored {restored} cached crates ({archive.stat().st_size >> 20} MiB) from {archive} in {time.time() - t:.0f} s")
else:
    restored = 0
    print(f"no cache on Drive yet; section 6 will save this session's builds to {archive}")


## 4. Point `%%rust` at the pinned toolchain and the backend

The same cell as in 04. `:cache 2000` turns on evcxr's compile cache with a 2000 MiB cap; it keeps a crate only when rustc built it without `-C incremental`, which the setup script arranged for all dependencies in `~/.cargo/config.toml`, while the cell itself stays incremental and uncached (do not add `CARGO_INCREMENTAL=0`: it would make cells cacheable too, and the next cell would load the previous one's build). With section 3's cache in place this cell takes a fraction of its cold time; the `Compiled N crates` line still counts the crates cargo went through.


In [ ]:
%%rust
:toolchain 1.99.1
:opt 3
:cache 2000
:build_env RUSTC_BOOTSTRAP=1
:build_env CUDA_HOME=/usr/local/cuda-13
:build_env CUDA_TOOLKIT_PATH=/usr/local/cuda-13
:build_env CUDA_OXIDE_LLC=/opt/colab-rust/cuda-oxide/bin/llc
:build_env RUSTFLAGS=-Cprefer-dynamic -Zcodegen-backend=/opt/colab-rust/cuda-oxide/lib/librustc_codegen_cuda.so -Zmir-enable-passes=-JumpThreading -Zalways-encode-mir -Csymbol-mangling-version=v0 -Cdebug-assertions=off -Coverflow-checks=off
:dep cuda-core = { path = "/opt/colab-rust/cuda-oxide/src/cuda-core" }
:dep cuda-device = { path = "/opt/colab-rust/cuda-oxide/src/cuda-oxide/crates/cuda-device" }
:dep cuda-host = { path = "/opt/colab-rust/cuda-oxide/src/cuda-oxide/crates/cuda-host" }
println!("cuda-oxide configured for {}", env!("CUDA_OXIDE_TARGET", "no PTX target: run the setup cell first"));

## 5. A kernel, compiled and launched

The vecadd kernel from 04, to show the cached crates work. The cell itself is compiled every time.


In [ ]:
%%rust
use cuda_core::LaunchConfig;
use cuda_core::{CudaContext, DeviceBuffer};
use cuda_device::{DisjointSlice, cuda_module, kernel, thread};

#[cuda_module]
mod kernels {
    use super::*;

    #[kernel]
    pub fn vecadd(a: &[f32], b: &[f32], mut c: DisjointSlice<f32>) {
        let idx = thread::index_1d();
        let raw = idx.get();
        if let Some(out) = c.get_mut(idx) {
            *out = a[raw] + b[raw];
        }
    }
}

let ctx = CudaContext::new(0).unwrap();
let stream = ctx.default_stream();
let loaded: kernels::LoadedModule = kernels::load(&ctx).unwrap();

const N: usize = 1024;
let a: Vec<f32> = (0..N).map(|i| i as f32).collect();
let b: Vec<f32> = (0..N).map(|i| (i * 2) as f32).collect();
let a_dev = DeviceBuffer::from_host(&stream, &a).unwrap();
let b_dev = DeviceBuffer::from_host(&stream, &b).unwrap();
let mut c_dev = DeviceBuffer::<f32>::zeroed(&stream, N).unwrap();
unsafe { loaded.vecadd(&stream, LaunchConfig::for_num_elems(N as u32), &a_dev, &b_dev, &mut c_dev) }.unwrap();
println!("{:?}", &c_dev.to_host_vec(&stream).unwrap()[..5]);

## 6. Save the compile cache to Drive

Run this last. It archives `~/.cache/evcxr` to the file section 3 reads, only when this session compiled something the archive does not have yet, and then flushes Drive so the write is complete before the runtime goes away. Section 3 skipped? This cell mounts Drive itself. It also picks up a cache that 04 built in this session, so you can decide to keep it after seeing how long the configuration cell took.


In [ ]:
from google.colab import drive
import pathlib, subprocess, time
oxide = pathlib.Path('/opt/colab-rust/cuda-oxide')
archive = pathlib.Path('/content/drive/MyDrive/colab-rust') / f"evcxr-cache-{oxide.resolve().name[:12]}-{(oxide / 'target').read_text().strip()}.tar"
cache = pathlib.Path.home() / '.cache' / 'evcxr'
entries = sum(1 for _ in cache.iterdir()) if cache.is_dir() else 0
if entries <= globals().get('restored', 0):
    print(f"nothing new to save ({entries} cached crates, all from Drive)")
else:
    drive.mount('/content/drive')
    archive.parent.mkdir(parents=True, exist_ok=True)
    t = time.time()
    tmp = archive.with_suffix('.tar.part')
    subprocess.run(['tar', '-cf', tmp, '-C', cache.parent, 'evcxr'], check=True)
    tmp.replace(archive)
    print(f"saved {entries} cached crates ({archive.stat().st_size >> 20} MiB) to {archive} in {time.time() - t:.0f} s; flushing Drive...")
    drive.flush_and_unmount()
    print("done; Drive is unmounted, run section 3 again if you need it in this session")


## Things to know

- **Cache stats.** A `%%rust` cell containing just `:cache` prints the entries, disk use and hits; `:clear_cache` empties the local cache (the archive on Drive stays until section 6 overwrites it).
- **One archive per pin and card.** The archive is named after the cuda-oxide commit and the PTX target; a pin bump or another GPU means one more cold session, and old archives can be deleted from `MyDrive/colab-rust` by hand.
- **Why a tar file and not a folder.** Drive's mount keeps no executable bit, so a build script copied back from a folder on Drive cannot run; the archive preserves it, and one large file is also what Drive handles well.
- **Everything else** (redefining modules, `%rust_reset`, other GPUs, what is pinned) is in 04's Things to know.
